# What a passenger and a tyre feel on the same road

A repeating surface profile produces a temporal excitation whose frequency depends on travel speed. Before running, predict whether doubling the road speed must increase the passenger acceleration. Change the first cell, rerun, and compare the two observables in the plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ms, mu = 250.0, 35.0   # sprung and unsprung masses, kg
ks, kt = 15000., 160000.  # suspension and tyre stiffnesses, N/m
c = 1200.               # suspension damping, N s/m
road_amplitude = .003   # m, sinusoidal height amplitude
wavelength = 4.0        # m per cycle
speed = 12.0            # m/s
assert min(ms, mu, ks, kt, c, road_amplitude, wavelength, speed) > 0

## Find the two undamped modes
Mass-normalising the stiffness produces a symmetric eigenproblem. The eigenvectors in these normalised coordinates must be mapped back to physical displacements. In an environment with OneSimulation this cell uses its symmetric eigensystem solver.

In [ ]:
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import symmetric_eigensystem
except ImportError:
    symmetric_eigensystem = np.linalg.eigh
K = np.array([[ks,-ks],[-ks,ks+kt]])
Minvhalf = np.diag([ms**-.5,mu**-.5])
omega2, modes = symmetric_eigensystem(Minvhalf @ K @ Minvhalf)
natural = np.sqrt(omega2)/(2*np.pi)
physical = Minvhalf @ modes
print("Undamped frequencies (Hz):", np.round(natural,3))
print("Body displacement / wheel displacement in each mode:", np.round(physical[0]/physical[1],3))
assert np.all(omega2 > 0)

## Turn a road height into body acceleration and tyre force
Complex displacements below are per unit road amplitude. The tyre force is determined by the difference between wheel and road positions, so its phase matters.

In [ ]:
def response(freq, damping):
    w = 2*np.pi*np.asarray(freq)
    s = ks + 1j*w*damping
    a, b = s - ms*w*w, s + kt - mu*w*w
    det = a*b - s*s
    return s*kt/det, a*kt/det

def observables(freq, damping):
    body, wheel = response(freq,damping)
    acc = (2*np.pi*freq)**2*np.abs(body)*road_amplitude
    force = kt*np.abs(wheel-1)*road_amplitude
    return acc,force

physical_frequency = speed/wavelength
a0, f0 = observables(physical_frequency,c)
print(f"At {physical_frequency:.3f} Hz: body acceleration amplitude {a0:.3f} m/s², dynamic tyre force amplitude {f0:.1f} N")
assert np.allclose(response(0., c), [1,1])
assert np.allclose(response(physical_frequency,c), np.linalg.solve(
    K-(2*np.pi*physical_frequency)**2*np.diag([ms,mu])+1j*2*np.pi*physical_frequency*c*np.array([[1,-1],[-1,1]]),[0,kt]))

## Compare damping for the same input
Follow the two dashed vertical markers from the low- to the high-frequency mode. The solid and dashed responses differ only in the suspension damping.

In [ ]:
f = np.geomspace(.25,35.,550)
fig, ax = plt.subplots(1,2,figsize=(9.2,3.6),layout="constrained")
for factor,style in [(1,"-"),(2,"--")]:
    a,force = observables(f,c*factor)
    ax[0].loglog(f,a,style,label=f"c = {c*factor:.0f} N s/m")
    ax[1].loglog(f,force,style,label=f"c = {c*factor:.0f} N s/m")
for p in ax:
    for mode in natural: p.axvline(mode,color="0.65",lw=.8,ls=":")
    p.set_xlabel("Road excitation (Hz)"); p.grid(which="major",alpha=.24); p.legend(fontsize=8)
ax[0].set(ylabel="Acceleration amplitude (m/s²)",title="Sprung mass")
ax[1].set(ylabel="Dynamic contact force amplitude (N)",title="Tyre against the road")
plt.show()

## Investigate
Find a frequency band where greater damping increases passenger acceleration. Translate each natural frequency to a road speed at the chosen wavelength. Check the maximum dynamic tyre force against the static corner load $(m_s+m_u)g$. If their ratio exceeds one, the linear model predicts tensile contact during part of a cycle and its contact assumption has failed.